In [1]:
import os
import pandas as pd
from pprint import pprint
import json
import pickle
from pyarrow.parquet import ParquetFile
import pyarrow as pa
import boto3
import numpy as np
import io

### Functions

In [2]:
# download from s3
def download_from_s3(str_local_path, str_bucket_path, str_project):
    # download file
    boto3.client('s3').download_file(str_project, str_bucket_path, str_local_path)

### Constants

In [3]:
int_n_debtors_target = 1
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
str_dirname_input = './input'
str_dirname_output = './output'
str_bigaccountid = 'ACCOUNTID'
str_bigdebtorid = 'DEBTORID'
int_bitdebtor = 0 # because it will be the codebtor
str_uniqueid = f'{str_bigaccountid}+{str_bigdebtorid}+{int_bitdebtor}'
print(f'Unique ID: {str_uniqueid}')
str_variant = 'noPTImodel10'

Project: 20231010-gen-xii
Unique ID: ACCOUNTID+DEBTORID+0


### Create input directory

In [4]:
# create input folder
try:
    os.mkdir(str_dirname_input)
except FileExistsError:
    pass

### Get a sample payload with 1 debtor

In [5]:
# load in gen xi requests
str_filename = 'df_genxi_2021-11-05_2021-11-09.csv' # this is different from the debtor in the 01_single script
str_uri = f's3://20221019-parse-all-gen-xi-payloads/01_pull_all_requests/{str_filename}'
df = pd.read_csv(str_uri, usecols=['strRequest'], nrows=100)

# find the index of a single debtor request
int_n_debtors = 0
a = 0
list_a = []
while int_n_debtors < int_n_debtors_target:
    # get int_n_debtors
    int_n_debtors = len(eval(df['strRequest'].iloc[a])['rows'])
    # logic
    if int_n_debtors == int_n_debtors_target:
        list_a.append(a)
    # increase a
    a += 1
    
# get index
int_idx = list_a[0]
# save payload
dict_request = eval(df['strRequest'].iloc[int_idx])
# filename
str_filename = 'dict_payload_gen_xi_single.json'
str_local_path = f'{str_dirname_input}/{str_filename}'
with open(str_local_path, "w") as outfile:
    json.dump(dict_request, outfile)

# cleanup
del df

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:275: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


### Import ```dict_payload_gen_xi_single.json```

In [6]:
dict_request_old = json.load(open(str_local_path))

### Create output directory

In [7]:
str_dirname_output = './output'
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

### Create variant directory

In [8]:
try:
    os.mkdir(f'{str_dirname_output}/{str_variant}')
except FileExistsError:
    pass

### Get the features in all of the models

In [9]:
# AD
str_filename = 'df_cols_in_model.csv'
str_uri = f's3://{str_project}/01_ad/02_model/{str_variant}/02_model/01_lambda_get_starting_feats/{str_filename}'
list_cols_ad = list(pd.read_csv(str_uri)['feature'])
print(f'There are {len(list_cols_ad)} features in the AD model')

There are 210 features in the AD model


In [10]:
# PD
str_filename = 'df_cols_in_model.csv'
str_uri = f's3://{str_project}/02_pricing_pd/02_model/{str_variant}/02_model/01_lambda_get_starting_feats/{str_filename}'
list_cols_pd = list(pd.read_csv(str_uri)['feature'])
print(f'There are {len(list_cols_pd)} features in the PD model')

There are 122 features in the PD model


In [11]:
# LGD
str_filename = 'df_cols_in_model.csv'
str_uri = f's3://{str_project}/03_pricing_lgd/02_model/{str_variant}/02_model/01_lambda_get_starting_feats/{str_filename}'
list_cols_lgd = list(pd.read_csv(str_uri)['feature'])
print(f'There are {len(list_cols_lgd)} features in the LGD model')

There are 76 features in the LGD model


In [12]:
# combine
list_cols = list_cols_ad + list_cols_pd + list_cols_lgd
print(f'Before removing duplicate columns, there are {len(list_cols)} features total')
# rm dups
list_cols = list(dict.fromkeys(list_cols))
print(f'After removing duplicate columns, there are {len(list_cols)} features total')

Before removing duplicate columns, there are 408 features total
After removing duplicate columns, there are 323 features total


### Get raw features

In [13]:
list_cols_raw = [
    'intopenbktype__app',
    'vehiclemake__app',
    'vehiclemodel__app',
    'applicationdate__app', 
    'fltgrossmonthly__income_sum',
    'amtfinanced__app',
    'bookvalue__app',
    'fltgrossmonthly__income_count',
    'vehicleyear__app', # for vehicle age
    'dealerstampcreation__app', # for dealership age
    'intterm__app',
    'fltapproveddowntotal__app', # for counter offers
    'intservicecontractmileage__app',
    'fltapprovedservicecontract__app',
    'fltdowncash__app',
    'fltgapinsurance__app',
    'miles_odometer__app',
    'fltadvance__app',
]
# extend
list_cols_raw = list_cols + list_cols_raw
# rm eng
list_cols_raw = [col for col in list_cols_raw if 'ENG-' not in col]

# rm dups
list_cols_raw = list(dict.fromkeys(list_cols_raw))
print(f'There are {len(list_cols_raw)} raw features in the model')

There are 331 raw features in the model


### Import data

In [14]:
%%time

# get data
str_filename = 'df_train_raw.gzip'
str_local_path = f'{str_dirname_output}/{str_filename}'
download_from_s3(
    str_local_path=str_local_path, 
    str_bucket_path=f'02_pricing_pd/01_data_prep/03_train_valid_test_split/{str_filename}', 
    str_project=str_project,
)
cls_file_parquet = ParquetFile(str_local_path)
first_n_rows = next(cls_file_parquet.iter_batches(batch_size=int_n_debtors_target))
df = pa.Table.from_batches([first_n_rows]).to_pandas()
os.remove(str_local_path)

# subset
df = df[list_cols_raw]
df.replace(['NaN','nan','None',None], np.nan, inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Rows: {df.shape[0]}; Columns: {df.shape[1]}')

# show
df

Rows: 1; Columns: 331
CPU times: user 3.09 s, sys: 640 ms, total: 3.73 s
Wall time: 1.13 s


<timed exec>:16: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


,bankruptcycount24month__ln,inquiryshortterm12month__ln,re01s__tu,bankruptcystatus__ln,bankruptcytimenewest__ln,fltgrossmonthly__income_sum,linka006__tu,trv10__tu,linka026__tu,balmag01__tu,...,vehiclemodel__app,applicationdate__app,amtfinanced__app,bookvalue__app,vehicleyear__app,dealerstampcreation__app,intterm__app,intservicecontractmileage__app,fltapprovedservicecontract__app,fltgapinsurance__app
0,0.0,1.0,0.0,0.0,-1.0,3875.850098,6.0,11.0,5.0,45.0,...,NaN,2013-10-01 07:28:54.833,14532.9,10550.0,2009,2012-06-18 09:31:54.393,66,0.0,0.0,0.0


### Application

In [15]:
# get format of table in request
dict_source_app_old = dict_request_old['rows'][0]['sources'][0]
dict_source_app_old

{'name': 'Application',
 'version': 1.0,
 'format': 'csv',
 'values': 'UniqueID,bigAccountId,bigDebtorId,bitDebtor,strCity,strName,strZipCode,ApplicationDate,bitApproved,bitSystemDecline,ApprovalDate,bitFunded,FundedDate,dtmStampCreation,dtmApproved,dtmDeclined,dtmFunded,defaultDate,ChargeOffDate,defaultAmount,chargeoffamount,ApplicationMonth,ApplicationQuarter,ApplicationDayofWeek,bigDealerID,bitRolled,bigDealerTypeId,DealerState,bitLHMGroup,DealerCity,DealerZip,bitDealerApplicantSameState,bitDealerApplicantSameCity,bitDealerApplicantSameZip,strDealershipTrackerType,intType,fltAcquisitionFee,fltAddFee,fltAllowance,fltAmountFinanced,fltApprovedAPR_contract,fltApprovedDebtToIncome,fltApprovedDownTotal,fltApprovedLoanToValue,fltApprovedPayment,fltApprovedPriceWholesale,fltApprovedServiceContract,fltDocumentFee,fltDownCash,fltGapInsurance,fltInsuredDisabilityAmount,fltInsuredDisabilityPremium,fltInsuredLifeAmount,fltInsuredLifePremium,fltInsuredUnemploymentAmount,fltInsuredUnemploymentPre

In [16]:
# create df with ids
df_tmp = pd.DataFrame({
    'uniqueid__app': [str_uniqueid],
    'bigaccountid__app': [str_bigaccountid],
    'bigdebtorid__app': [str_bigdebtorid],
})

# get application cols
list_cols = [col for col in df.columns if '__app' in col]

print(f'There are {len(list_cols)} application features:')
for a, col in enumerate(list_cols):
    print(f'{a+1} - {col}')

There are 22 application features:
1 - dealerstate__app
2 - strdealershiptrackertype__app
3 - bitdebtor__app
4 - strname__app
5 - intopenbktype__app
6 - bitdealertrack__app
7 - vehiclemake__app
8 - fltadvance__app
9 - bitgap__app
10 - fltdowncash__app
11 - fltapproveddowntotal__app
12 - miles_odometer__app
13 - vehiclemodel__app
14 - applicationdate__app
15 - amtfinanced__app
16 - bookvalue__app
17 - vehicleyear__app
18 - dealerstampcreation__app
19 - intterm__app
20 - intservicecontractmileage__app
21 - fltapprovedservicecontract__app
22 - fltgapinsurance__app


In [17]:
# concat
df_tmp = pd.concat([df_tmp, df[list_cols]], axis=1)

# rm app
df_tmp.columns = [col.split('__')[0] for col in df_tmp.columns]
pprint(list(df_tmp.columns))
print('')

# bitdebtor
df_tmp['bitdebtor'] = int_bitdebtor

# show
df_tmp

['uniqueid',
 'bigaccountid',
 'bigdebtorid',
 'dealerstate',
 'strdealershiptrackertype',
 'bitdebtor',
 'strname',
 'intopenbktype',
 'bitdealertrack',
 'vehiclemake',
 'fltadvance',
 'bitgap',
 'fltdowncash',
 'fltapproveddowntotal',
 'miles_odometer',
 'vehiclemodel',
 'applicationdate',
 'amtfinanced',
 'bookvalue',
 'vehicleyear',
 'dealerstampcreation',
 'intterm',
 'intservicecontractmileage',
 'fltapprovedservicecontract',
 'fltgapinsurance']



,uniqueid,bigaccountid,bigdebtorid,dealerstate,strdealershiptrackertype,bitdebtor,strname,intopenbktype,bitdealertrack,vehiclemake,...,vehiclemodel,applicationdate,amtfinanced,bookvalue,vehicleyear,dealerstampcreation,intterm,intservicecontractmileage,fltapprovedservicecontract,fltgapinsurance
0,ACCOUNTID+DEBTORID+0,ACCOUNTID,DEBTORID,Ohio,Franchise,0,Ohio,NaN,1.0,Chevrolet,...,NaN,2013-10-01 07:28:54.833,14532.9,10550.0,2009,2012-06-18 09:31:54.393,66,0.0,0.0,0.0


In [18]:
# get strings
cls_str_io = io.StringIO()
df_tmp.to_csv(cls_str_io, index=False)
str_df_tmp = cls_str_io.getvalue()

# make dict source
dict_source_app = {
    'name': 'Application',
    'version': 1.0,
    'format': 'csv',
    'values': str_df_tmp,
}
pprint(dict_source_app)

{'format': 'csv',
 'name': 'Application',
 'values': 'uniqueid,bigaccountid,bigdebtorid,dealerstate,strdealershiptrackertype,bitdebtor,strname,intopenbktype,bitdealertrack,vehiclemake,fltadvance,bitgap,fltdowncash,fltapproveddowntotal,miles_odometer,vehiclemodel,applicationdate,amtfinanced,bookvalue,vehicleyear,dealerstampcreation,intterm,intservicecontractmileage,fltapprovedservicecontract,fltgapinsurance\n'
           'ACCOUNTID+DEBTORID+0,ACCOUNTID,DEBTORID,Ohio,Franchise,0,Ohio,,1.0,Chevrolet,1.1066350710900474,0,2000.0,2000.0,46429.0,,2013-10-01 '
           '07:28:54.833,14532.9,10550.0,2009,2012-06-18 '
           '09:31:54.393,66,0.0,0.0,0.0\n',
 'version': 1.0}


### Income

In [19]:
# get format of table in request
dict_source_income_old = dict_request_old['rows'][0]['sources'][1]
dict_source_income_old

{'name': 'Incomes',
 'version': 1.0,
 'format': 'csv',
 'values': 'UniqueID,bigAccountId,bigDebtorId,bigIncomeId,bigIncomeTypeId,bitCurrent,bitEmployment,bitFullTime,bitInvalid,bitPulledFromBureau,bitSelfEmployed,bitUse,bitVerified,dtmEnd,dtmStampCreation,dtmStart,dtmVerified,fltGrossMonthly,intMonths,intWeeklyHour,intYears,strComment,strCommentVerification,strEmployerAddress,strEmployerBusiness,strEmployerContact,strEmployerName,strOccupation,strPhone,strPhoneDepartment,strPhoneFax,strPhonePersonnel,strPosition,strSource,strSpokeTo,strType,strWageType,dtmDateOnPayStub,flt401kLoanPayment,fltChildSupport,fltGarnish,bigEmployeeId_verified,bitBrokerage,bigDealerContactId_createdBy,bitW2,bit1099,strFrequency,bitRequirePOI,strPhoneExt,strDepartmentExt,strPersonnelExt,strDepartment,strWeekly,strEveryOther,strMonthly,strPayDays,dtmIncomeVerified,bigEmployeeId_verifiedAmount,fltYTDPaystub,fltGrossPaystub,fltBasePaystub,fltRatePaystub,bitIncomeVerified,bitIncomeVerifiedCollections\n5817564__730

In [20]:
# create df with ids
df_tmp = pd.DataFrame({
    'uniqueid': [str_uniqueid],
    'bigaccountid': [str_bigaccountid],
    'bigdebtorid': [str_bigdebtorid],
})

# convert to df
df_tmp_income = pd.read_csv(io.StringIO(dict_source_income_old['values']), delimiter=',', usecols=['bitUse','bitInvalid','fltGrossMonthly'])
print(f'There are {df_tmp_income.shape[1]} income features:')

# concat
df_tmp = pd.concat([df_tmp, df_tmp_income], axis=1)

# lower
df_tmp.columns = [col.lower() for col in df_tmp.columns]

# show
df_tmp

There are 3 income features:


,uniqueid,bigaccountid,bigdebtorid,bitinvalid,bituse,fltgrossmonthly
0,ACCOUNTID+DEBTORID+0,ACCOUNTID,DEBTORID,False,True,3120


In [21]:
# get strings
cls_str_io = io.StringIO()
df_tmp.to_csv(cls_str_io, index=False)
str_df_tmp = cls_str_io.getvalue()

# make dict source
dict_source_income = {
    'name': 'Incomes',
    'version': 1.0,
    'format': 'csv',
    'values': str_df_tmp,
}
pprint(dict_source_income)

{'format': 'csv',
 'name': 'Incomes',
 'values': 'uniqueid,bigaccountid,bigdebtorid,bitinvalid,bituse,fltgrossmonthly\n'
           'ACCOUNTID+DEBTORID+0,ACCOUNTID,DEBTORID,False,True,3120\n',
 'version': 1.0}


### Lexis Nexis

In [22]:
# get format of table in request
dict_source_ln_old = dict_request_old['rows'][0]['sources'][3]
dict_source_ln_old

{'name': 'Lexis Nexis Risk View 5',
 'version': 1.0,
 'format': 'csv',
 'values': 'UniqueID,bigLNRiskViewAttributesV5id,bigAccountId,bigDebtorId,bigLNRiskViewScoreId,bitInvalid,dtmStampCreation,attribute_index,inputprovidedfirstname,inputprovidedlastname,inputprovidedstreetaddress,inputprovidedcity,inputprovidedstate,inputprovidedzipcode,inputprovidedssn,inputprovideddateofbirth,inputprovidedphone,inputprovidedlexid,subjectrecordtimeoldest,subjectrecordtimenewest,subjectnewestrecord12month,subjectactivityindex03month,subjectactivityindex06month,subjectactivityindex12month,subjectage,subjectdeceased,subjectssncount,subjectstabilityindex,subjectstabilityprimaryfactor,subjectabilityindex,subjectabilityprimaryfactor,subjectwillingnessindex,subjectwillingnessprimaryfactor,confirmationsubjectfound,confirmationinputname,confirmationinputdob,confirmationinputssn,confirmationinputaddress,sourcenonderogprofileindex,sourcenonderogcount,sourcenonderogcount03month,sourcenonderogcount06month,sourcen

In [23]:
# create df with ids
df_tmp = pd.DataFrame({
    'uniqueid__app': [str_uniqueid],
    'bigaccountid__app': [str_bigaccountid],
    'bigdebtorid__app': [str_bigdebtorid],
})

# get ln cols
list_cols = [col for col in df.columns if '__ln' in col]
print(f'There are {len(list_cols)} lexis nexis features:')
for a, col in enumerate(list_cols):
    print(f'{a+1} - {col}')

# concat
df_tmp = pd.concat([df_tmp, df[list_cols]], axis=1)

# rm ln
df_tmp.columns = [col.split('__')[0] for col in df_tmp.columns]

# show
df_tmp

There are 45 lexis nexis features:
1 - bankruptcycount24month__ln
2 - inquiryshortterm12month__ln
3 - bankruptcystatus__ln
4 - bankruptcytimenewest__ln
5 - addrinputtimenewest__ln
6 - derogseverityindex__ln
7 - bankruptcychapter__ln
8 - inquirytelcom12month__ln
9 - addronfilecount__ln
10 - inquirybanking12month__ln
11 - subjectactivityindex06month__ln
12 - subjectwillingnessindex__ln
13 - criminalnonfelonycount__ln
14 - evictioncount12month__ln
15 - subjectabilityindex__ln
16 - ssnproblems__ln
17 - criminalfelonycount__ln
18 - sourcenonderogcount12month__ln
19 - assetproppurchasetimeoldest__ln
20 - evictioncount__ln
21 - assetindex__ln
22 - educationattendance__ln
23 - assetproppurchasetimenewest__ln
24 - assetpropnewestsaleprice__ln
25 - sourcenonderogcount03month__ln
26 - addrinputdelivery__ln
27 - alertregulatorycondition__ln
28 - subjectactivityindex03month__ln
29 - bankruptcycount__ln
30 - addrinputavmvalue12month__ln
31 - addrinputownershipindex__ln
32 - sourcenonderogcount__ln
3

,uniqueid,bigaccountid,bigdebtorid,bankruptcycount24month,inquiryshortterm12month,bankruptcystatus,bankruptcytimenewest,addrinputtimenewest,derogseverityindex,bankruptcychapter,...,evictiontimenewest,addrinputphoneservice,addrcurrentsubjectowned,addrpreviouslengthofres,lienjudgmenttimenewest,businessassociationtimeoldest,addrinputproblems,assetpropevercount,addrpreviousdwelltype,addrinputmatchindex
0,ACCOUNTID+DEBTORID+0,ACCOUNTID,DEBTORID,0.0,1.0,0.0,-1.0,2.0,4.0,0.0,...,32.0,1.0,0.0,59.0,15.0,-1.0,0.0,1.0,S,1.0


In [24]:
# get strings
cls_str_io = io.StringIO()
df_tmp.to_csv(cls_str_io, index=False)
str_df_tmp = cls_str_io.getvalue()

# make dict source
dict_source_ln = {
    'name': 'Lexis Nexis Risk View 5',
    'version': 1.0,
    'format': 'csv',
    'values': str_df_tmp,
}
pprint(dict_source_ln)

{'format': 'csv',
 'name': 'Lexis Nexis Risk View 5',
 'values': 'uniqueid,bigaccountid,bigdebtorid,bankruptcycount24month,inquiryshortterm12month,bankruptcystatus,bankruptcytimenewest,addrinputtimenewest,derogseverityindex,bankruptcychapter,inquirytelcom12month,addronfilecount,inquirybanking12month,subjectactivityindex06month,subjectwillingnessindex,criminalnonfelonycount,evictioncount12month,subjectabilityindex,ssnproblems,criminalfelonycount,sourcenonderogcount12month,assetproppurchasetimeoldest,evictioncount,assetindex,educationattendance,assetproppurchasetimenewest,assetpropnewestsaleprice,sourcenonderogcount03month,addrinputdelivery,alertregulatorycondition,subjectactivityindex03month,bankruptcycount,addrinputavmvalue12month,addrinputownershipindex,sourcenonderogcount,criminalnonfelonytimenewest,addrlastmovetaxratiodiff,addrcurrentdwelltype,evictiontimenewest,addrinputphoneservice,addrcurrentsubjectowned,addrpreviouslengthofres,lienjudgmenttimenewest,businessassociationtimeoldest

### TransUnion

In [25]:
# get format of table in request
dict_source_tu_old = dict_request_old['rows'][0]['sources'][4]

# make dict source
dict_source_tu = {
    'name': 'TUXML',
    'version': 1.0,
    'format': 'xml',
    'values': dict_source_tu_old['values'],
}
#pprint(dict_source_tu)

## Create payload

In [26]:
# list of sources
list_sources = [
    dict_source_app,
    dict_source_income,
    dict_source_ln,
    dict_source_tu,
]

# make dict row
dict_row_codebtor = {
    'row_id': str_uniqueid,
    'sources': list_sources,
}
#pprint(dict_row_codebtor)

In [27]:
# load single payload from preview notebook
str_filename = 'dict_payload_gen_xii_single.json'
str_local_path = f'../01_single/output/{str_variant}/{str_filename}'
dict_request_new = json.load(open(str_local_path))

# get list of rows
list_rows = dict_request_new['rows'].copy()
# append
list_rows.append(dict_row_codebtor)
# assign
dict_request_new['rows'] = list_rows
#pprint(dict_request_new)

In [28]:
# filename
str_filename = 'dict_payload_gen_xii_codebtor.json'
str_local_path = f'{str_dirname_output}/{str_variant}/{str_filename}'
with open(str_local_path, "w") as outfile:
    json.dump(dict_request_new, outfile)
#pprint(dict_payload)